# Cell 1: Set up the Notebook Environment

This cell installs the required library, connects Google Drive, and checks that
a GPU is available before we start the main processing.

In [ ]:
# Install the sentence-transformers library used later in the notebook for generating and comparing text embeddings.
!pip install sentence-transformers -q

# Connect Google Colab to Google Drive so the notebook can access
# the project files and datasets stored there.
from google.colab import drive
drive.mount('/content/drive')

# check GPU is actually on
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - fix Runtime type")

Mounted at /content/drive
GPU available: True | Tesla T4


# Cell 2: Quick Required Project Files

Before running the main pipeline, confirm that the reducer and preference-model files are available in the current Colab environment.

In [ ]:
# These two files are used later in the notebook.
# Check that they are available before continuing, so a missing file is caught early rather than causing a confusing error later.
# The reducer file is from Michael, and the d5_preference_model is from Wesley. During testing, if missing these files check their repo or ask them.

import os
for f in ["reducer_v0_1_1.py", "d5_preference_model.py"]:
    print(f, "->", "FOUND" if os.path.exists(f) else "MISSING - upload it")

reducer_v0_1_1.py -> FOUND
d5_preference_model.py -> FOUND


# Cell 3: Pre-Flight Test - Check Activity Matching

Before loading the full corpus, use a real record from the corpus to check that activity values can be read correctly and that the activity matcher only searches the intended `prefLabel` values.

In [ ]:
import json
from pathlib import Path
from reducer_v0_1_1 import _norm

# Location of the corpus used by the rest of the Notebook
CORPUS_DIR = "/content/drive/MyDrive/S1_CORPUS_V2026-08-27"     # Change this based on the output of Michael's S0/S1 reconstruction process


# Extract the human-readable prefLabel values from an activity field. The corpus can store activities as either a single object/string or a list.
def _pref_labels(activity_value):
    if not activity_value:
        return []

    # Assigns the activities as either a single object/string or a list.
    acts = (
        activity_value
        if isinstance(activity_value, list)
        else [activity_value]
    )

    labels = []

    for a in acts:
        if isinstance(a, dict):
            pl = a.get("prefLabel")
            if pl:
                labels.append(str(pl))

        elif isinstance(a, str):
            labels.append(a)

    return labels

# Use the same text normalisation as the reducer so that the test behaves consistently with the main matching logic.
def _norm_text(value):
    return _norm(value)

# Create a matcher for an activity term. Matching is deliberately limited to prefLabel values rather than searching URLs or other fields associated with the activity.
def activity_matches(query_term):
    q = _norm_text(query_term).strip()

    def pred(activity_value):
        return any(
            q in _norm_text(label)
            for label in _pref_labels(activity_value)
        )

    return pred


# Find the provider files in the corpus.
files = sorted(
    Path(CORPUS_DIR).glob("*/s1_records.jsonl")
)

if not files:
    raise FileNotFoundError(
        f"No s1_records.jsonl files found under:\n{CORPUS_DIR}"
    )

print(f"Found {len(files)} provider files.")


# Look through the real corpus until we find a record containing a non-empty activity field.
rec = None
test_file = None
records_checked = 0

for rf in files:
    if rf.stat().st_size == 0:
        continue

    with open(rf, encoding="utf-8") as f:
        for line in f:
            try:
                candidate = json.loads(line)
            except json.JSONDecodeError:
                continue

            records_checked += 1
            fields = candidate.get("fields", {}) or {}
            activity = fields.get("activity")

            if activity:
                rec = candidate
                test_file = rf
                break

        if rec is not None:
            break

if rec is None:
    raise RuntimeError(
        f"No record containing a non-empty 'activity' field "
        f"was found after checking {records_checked:,} records."
    )

print(
    f"Found activity-bearing record after checking "
    f"{records_checked:,} records."
)

print(f"Testing with: {test_file}")

# Display the actual activity data found in the corpus.
fields = rec.get("fields", {}) or {}
activity = fields.get("activity")

print("\nREAL activity field:")
print(activity)


# Extract and display the human-readable activity labels.
labels = _pref_labels(activity)

print("\nExtracted prefLabels:")
for label in labels:
    print(f"  - {label}")


# Test the expected "swimming" query against the real activity data.
swimming_match = activity_matches("swimming")(activity)

print("\n'swimming' matches?:", swimming_match)


# Try a few short terms that could accidentally match other text. These checks help confirm that matching is happening against prefLabel values only.
print("\nCollision checks:")

for term in ["list", "open", "io"]:
    result = activity_matches(term)(activity)
    print(f"  '{term}' matches?: {result}")


# Summarise what the pre-flight test demonstrated.
print("\n" + "=" * 60)

if swimming_match:
    print("✅ PRE-FLIGHT TEST PASSED")
    print("   Found a real activity-bearing record.")
    print("   prefLabels were extracted.")
    print("   'swimming' matched a real prefLabel.")
    print("   Activity matching is restricted to prefLabel.")
    print("   @id / inScheme URL text is not searched.")
    print("   Unicode-normalised matching is being used.")
else:
    print("⚠️ PRE-FLIGHT TEST PARTIALLY PASSED")
    print("   Found a real activity-bearing record.")
    print("   prefLabels were extracted.")
    print("   But this particular record does not contain")
    print("   'swimming' in its prefLabels.")
    print("   Activity matching is still restricted to prefLabel.")
    print("   @id / inScheme URL text is not searched.")
    print("   Unicode-normalised matching is being used.")

print("=" * 60)

print("\nCell 4 can now perform the corpus load.")

Found 66 provider files.
Found activity-bearing record after checking 2,368 records.
Testing with: /content/drive/MyDrive/S1_CORPUS_V2026-08-27/Better/s1_records.jsonl

REAL activity field:
[{'@type': 'Concept', '@id': 'https://openactive.io/activity-list#2750229d-b725-4171-9276-376be913957c', 'inScheme': 'https://openactive.io/activity-list', 'prefLabel': 'Swimming'}]

Extracted prefLabels:
  - Swimming

'swimming' matches?: True

Collision checks:
  'list' matches?: False
  'open' matches?: False
  'io' matches?: False

✅ PRE-FLIGHT TEST PASSED
   Found a real activity-bearing record.
   prefLabels were extracted.
   'swimming' matched a real prefLabel.
   Activity matching is restricted to prefLabel.
   @id / inScheme URL text is not searched.
   Unicode-normalised matching is being used.

Cell 4 can now perform the corpus load.


# Cell 4: Full Corpus Retrieval and Evidence Gating

This cell runs the main retrieval pipeline over the reconstructed OpenActive corpus. It:
- loads the full corpus and records basic accounting information;
- builds searchable cards from the raw records;
- retrieves relevant records using both BM25 and dense semantic search;
- combines the two retrieval rankings using Reciprocal Rank Fusion (RRF);
- applies the evidence-gating reducer to check whether each requested constraint is actually supported;
- applies the preference model to help order the resulting candidates; and
- saves a summary of the test results to Google Drive.

The important distinction is that **retrieval finds potentially relevant records, while the evidence gate decides what can actually be treated as supported**.

The cell also checks for excluded records and duplicate retrieval IDs so that corpus-loading problems are visible rather than silently ignored.

No artificial provider or corpus-size limit is applied in this run.

In [ ]:
import json
import re
import math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

# Existing project components:
# reducer_v0_1_1 handles evidence gating (Michael's)
# d5_preference_model provides the preference score (Wesley's)
from reducer_v0_1_1 import (
    reduce_decision,
    match_text,
    match_date_window,
    _norm
)

from d5_preference_model import (
    PreferenceBelief,
    preference_score
)

# ============================================================
# CONFIGURATION
# ============================================================
# Define where the reconstructed corpus and cached embeddings are stored. No artificial limit is applied to the corpus in this experiment.
CORPUS_DIR = "/content/drive/MyDrive/S1_CORPUS_V2026-08-27"
CACHE_DIR  = "/content/drive/MyDrive/d4_embeddings"

# None means the experiment uses the complete available corpus.
PER_PROVIDER_CAP = None
SKIP_GIANTS_OVER_MB = None

# ============================================================
# TEXT NORMALISATION
# ============================================================
# Use the same text normalisation as the reducer so that the test behaves consistently with the main matching logic.
def _norm_text(value):
    return _norm(value)


# ============================================================
# ACTIVITY HELPERS
# ============================================================
# Extract activity labels from an OpenActive activity field. Only prefLabel values are used for activity matching; category names and URLs are deliberately not searched.
def _pref_labels(activity_value):
    if not activity_value:
        return []

    acts = (
        activity_value
        if isinstance(activity_value, list)
        else [activity_value]
    )

    labels = []

    for a in acts:
        if isinstance(a, dict):
            pl = a.get("prefLabel")

            if pl:
                labels.append(str(pl))

        elif isinstance(a, str):
            labels.append(a)

    return labels

# Create a reusable activity-matching function for a user's search term. The comparison uses the project's normalised text representation and searches the extracted prefLabels.
def activity_matches(query_term):
    # Normalise the user's search term using the same rules used elsewhere in the retrieval/evidence pipeline.
    q = _norm_text(query_term).strip()

    def pred(activity_value):
        return any(
            q in _norm_text(label)
            for label in _pref_labels(activity_value)
        )

    return pred


# ============================================================
# FACET GENERATION
# ============================================================
# Do NOT derive the activity facet from category. Activity is the field used by the matcher and preference model.
def card_facets(f):
    import datetime

    activity_labels = _pref_labels(f.get("activity"))
    labels = " ".join(activity_labels)

    labels_norm = _norm_text(labels)

    # Create a small set of activity facets used by the preference model. These are derived from activity prefLabels, rather than from the separate category field.
    activity = (
    "swim"
    if "swim" in labels_norm
    else "fitness suite"
    if ("fitness" in labels_norm or "gym" in labels_norm)
    else "junior"
    if "junior" in labels_norm
    else "group exercise"
    if "group" in labels_norm
    else "leisure pool"
    if "pool" in labels_norm
    else None
    )

    day_t = None
    time_t = None

    sd = f.get("startDate")

    if sd:
        try:
            d = datetime.datetime.fromisoformat(sd)

            day_t = (
                "weekend"
                if d.weekday() >= 5
                else "weekday"
            )

            time_t = (
                "morning"
                if 5 <= d.hour < 12
                else "afternoon"
                if 12 <= d.hour < 17
                else "evening"
            )

        except (ValueError, TypeError):
            # Invalid dates should not stop the whole corpus load. The date/time facets are simply left unavailable.
            pass

    facets = {}

    if activity:
        facets["activity"] = activity

    if day_t:
        facets["day_type"] = day_t

    if time_t:
        facets["time"] = time_t

    return facets

# ============================================================
# SEARCH CARD CREATION
# ============================================================
# Convert a full corpus record into the smaller representation needed by the retrieval system while retaining the original record for later evidence checking.
def build_card(record):
    f = record.get("fields", {}) or {}

    name = f.get("name") or ""

    activity_labels = _pref_labels(
        f.get("activity")
    )

    loc = f.get("location") or {}

    loc_name = (
        loc.get("name", "")
        if isinstance(loc, dict)
        else ""
    )

    instr = f.get("attendeeInstructions") or ""

    # Build the text used by the retrieval systems from the fields that are relevant to the search task.
    text = " ".join(
        str(p)
        for p in [
            name,
            " ".join(activity_labels),
            loc_name,
            instr
        ]
        if p
    ).strip()

    # Use the record's OpenActive @id as the retrieval/fusion key. The fallbacks are only used if the normal @id is unavailable.
    record_id = (
        record.get("@id")
        or record.get("id")
        or record.get("item_id")
    )

    return {
        "record_id": record_id,
        "item_id": record.get("item_id"),
        "text": text,
        "name": name,
        "facets": card_facets(f),
        # Keep the original record so the evidence reducer can inspect the underlying fields rather than only the simplified search representation.
        "_record": record
    }


# ============================================================
# FULL CORPUS LOADING
# ============================================================
# Load all provider record files from the reconstructed corpus. The function keeps track of searchable records, excluded
# records, exclusion reasons, and duplicate retrieval IDs so that the corpus accounting can be checked explicitly.
def load_corpus_cards():
    files = sorted(
        Path(CORPUS_DIR).glob("*/s1_records.jsonl")
    )

    if not files:
        raise FileNotFoundError(
            f"No s1_records.jsonl files found under:\n{CORPUS_DIR}"
        )

    cards = []
    report = {}
    total_records = 0
    loaded_records = 0
    excluded_records = 0
    excluded_reasons = Counter()

    # Track duplicate retrieval IDs explicitly rather than allowing collisions to remain hidden.
    seen_record_ids = set()
    duplicate_record_ids = Counter()

    for rf in files:
        prov = rf.parent.name
        size_mb = rf.stat().st_size / 1048576

        if rf.stat().st_size == 0:
            report[prov] = 0
            continue

        if (
            SKIP_GIANTS_OVER_MB
            and size_mb > SKIP_GIANTS_OVER_MB
        ):
            report[prov] = f"SKIP({size_mb:.0f}MB)"
            continue

        n = 0

        with open(rf, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                # Ignore malformed individual lines while allowing the remainder of the provider file to be processed.
                except json.JSONDecodeError:
                    continue

                total_records += 1
                c = build_card(rec)

                # Records without searchable text are still included in the overall corpus accounting, but cannot be meaningfully retrieved.
                if not c["text"]:
                    excluded_records += 1
                    excluded_reasons[
                        "no_searchable_text"
                    ] += 1
                    continue
                
                # Check whether the retrieval key has appeared before. This helps identify possible record-ID collisions.
                rid = c["record_id"]

                if rid in seen_record_ids:
                    duplicate_record_ids[rid] += 1
                seen_record_ids.add(rid)
                cards.append(c)
                n += 1
                loaded_records += 1
                
        report[prov] = n

    # Store the accounting information so that the corpus load can be checked after the function has finished.
    report["_total_records"] = total_records
    report["_loaded_records"] = loaded_records
    report["_excluded_records"] = excluded_records
    report["_excluded_reasons"] = dict(excluded_reasons)
    report["_duplicate_record_ids"] = dict(
        duplicate_record_ids
    )

    return cards, report

# ============================================================
# TEXT TOKENISATION
# ============================================================
# Simple tokenisation used by the keyword-based BM25 retriever.
_TOKEN = re.compile(r"[a-z0-9]+")

# Convert a piece of text into normalised search tokens for BM25.
def tokenize(s):
    return _TOKEN.findall(
        _norm_text(s)
    )


# ============================================================
# BM25 RETRIEVER
# ============================================================
# Implement the BM25 keyword-based retrieval model. BM25 ranks records according to how strongly their searchable text matches the terms in the query.
class BM25:
    def __init__(self, cards, k1=1.5, b=0.75):
        self.cards = cards
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(c["text"]) for c in cards]
        self.N = len(self.docs)
        self.avgdl = (sum(len(d) for d in self.docs) / max(1, self.N))

        df = Counter()

        for d in self.docs:
            for t in set(d):
                df[t] += 1

        # Calculate inverse-document-frequency values for the terms appearing in the corpus.
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}

        self.tf = [Counter(d) for d in self.docs]

    # Search the corpus using BM25 and return the highest-ranked record IDs together with their BM25 scores.
    def search(self, query, top_k=100):
        q = tokenize(query)
        out = []

        for i, tf in enumerate(self.tf):
            dl = len(self.docs[i])
            s = 0.0

            for term in q:
                if term not in tf:
                    continue

                f = tf[term]
                s += (self.idf.get(term, 0.0) * (f * (self.k1 + 1))/ (f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)))

            if s > 0:
                out.append((i, s))

        out.sort(key=lambda x: x[1], reverse=True)

        return [(self.cards[i]["record_id"], sc) for i, sc in out[:top_k]]


# ============================================================
# DENSE RETRIEVER
# ============================================================
# Implement semantic retrieval using sentence-transformer embeddings. Corpus embeddings are cached so that repeated experiments do not need to recompute them.
class DenseRetriever:
    def __init__(self, cards):
        from sentence_transformers import SentenceTransformer
        self.cards = cards
        self.ids = [c["record_id"] for c in cards]
        # Use the same sentence-transformer model for all corpus and query embeddings. GPU is used for the large corpus.
        self.model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

        Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

        # Cache embeddings using the corpus size so that repeated experiments can reuse previously calculated vectors.
        cache = (Path(CACHE_DIR) / f"corpus_{len(cards)}.npy")

        if (cache.exists() and np.load(cache).shape[0] == len(cards)):
            self.emb = np.load(cache)
            print("  loaded cached embeddings")
        else:
            print(
                f"  embedding {len(cards):,} "
                "records on GPU..."
            )

            emb = self.model.encode([c["text"] for c in cards], normalize_embeddings=True, batch_size=256, show_progress_bar=True, device="cuda")
            self.emb = np.asarray(emb, dtype=np.float32)
            np.save(cache, self.emb)


    # Embed a query and compare it against the cached corpus embeddings to produce a semantic similarity ranking.
    def search(self, query, top_k=100):
        # Embed the query using the same model and normalisation used for the corpus records.
        q = np.asarray(self.model.encode([query], normalize_embeddings=True, device="cuda"), dtype=np.float32)[0]
        # With normalised vectors, the dot product gives cosine similarity between the query and each corpus record.
        sims = self.emb @ q
        k = min(top_k, len(sims))
        idx = np.argpartition(-sims, k - 1)[:k]
        idx = idx[np.argsort(-sims[idx])]
        return [(self.ids[i], float(sims[i])) for i in idx]


# ============================================================
# RECIPROCAL RANK FUSION
# ============================================================
# Combine multiple retrieval rankings into a single ranking. This allows the BM25 and dense retrievers to contribute without requiring their scores to be directly comparable.
def rrf(rankings, k=60, top_k=100):
    # Combine the keyword and dense rankings without requiring their raw scores to be on the same numerical scale.
    fused = defaultdict(float)

    for r in rankings:
        for rank, (iid, _) in enumerate(r):
            fused[iid] += (1.0/ (k + rank + 1))

    return sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]


# ============================================================
# PRICE HELPERS
# ============================================================
# Check whether an OpenActive offers value contains at least one explicitly free option with a price of zero.
def price_free(o):
    o = (o if isinstance(o, list) else [o])

    return any(x.get("price") == 0 for x in o if (isinstance(x, dict) and x.get("price") is not None))

# Create a price-checking function that accepts an activity when its lowest available price is at or below the supplied limit.
def price_max(lim):
    def p(o):
        o = (o if isinstance(o, list) else [o])
        pr = [x.get("price") for x in o if (isinstance(x, dict) and x.get("price") is not None)]
        return bool(pr) and min(pr) <= lim
    return p


# ============================================================
# CONSTRAINT BUILDING
# ============================================================
# Translate the user's query into a set of evidence predicates. These predicates are passed to the reducer, which determines
# whether each requested condition is supported, refuted, or indeterminate.
def build_constraints(q):
    c = []
    # Each requested condition becomes an explicit predicate that the evidence reducer will evaluate against the record.
    if q.get("activity"):
        c.append(("activity", activity_matches(q["activity"])))

    if q.get("region"):
        c.append(("location", match_text(q["region"])))

    if q.get("free"):
        c.append(("offers", price_free))
    elif q.get("max_price") is not None:
        c.append(("offers", price_max(q["max_price"])))

    if (q.get("date_from") and q.get("date_to")):
        c.append(("startDate", match_date_window(q["date_from"], q["date_to"])))

    if q.get("level"):
        c.append(("level", match_text(q["level"])))

    if q.get("age"):
        c.append(("ageRange", match_text(q["age"])))

    return c


# ============================================================
# RECOMMENDATION PIPELINE
# ============================================================
# Run the complete recommendation process for one query: retrieve candidates, combine retrieval rankings, apply the evidence gate, calculate preference scores, and produce the
# final ordered results.
def recommend(query, cards, bm25, dense, belief=None, top_k=10):
    qt = query.get("activity") or ""
    cons = build_constraints(query)
    belief = (belief if belief is not None else PreferenceBelief())
    # Retrieve candidates using both keyword and semantic search, then combine their rankings with reciprocal rank fusion.
    fused = rrf([bm25.search(qt, 100), dense.search(qt, 100)],top_k=100)
    by_id = {c["record_id"]: c for c in cards}
    res = []

    for iid, rs in fused:
        card = by_id.get(iid)
        if card is None:
            continue
        # The evidence reducer is applied after retrieval. Retrieval relevance therefore does not by itself make a candidate an evidence-supported result.
        dec, detail = reduce_decision(card["_record"], cons)

        res.append(
            {
                "gate": dec,
                "record_id": card["record_id"],
                "item_id": card["item_id"],
                "name": card["name"],
                "startDate": (
                    card["_record"]
                    .get("fields", {})
                    .get("startDate")
                ),
                "preference": round(
                    preference_score(
                        card["facets"],
                        belief
                    ),
                    3
                ),
                "retrieval": round(
                    rs,
                    4
                ),
                "detail": detail
            }
        )

    # Evidence status is kept separate from preference and retrieval relevance. A high preference/retrieval score cannot turn an unsupported record into a supported match.
    order = {
        "supported_match": 0,
        "evidence_indeterminate": 1,
        "bounded_non_match": 2
    }

    res.sort(key=lambda r: (order.get(r["gate"], 3), -r["preference"], -r["retrieval"]))

    return res[:top_k]


# ============================================================
# INITIALISE THE PIPELINE
# ============================================================
# Confirm that the function definitions have loaded and display the corpus configuration before starting the expensive parts of the experiment.
print("Cell 4 function definitions loaded.")
print("\nFull corpus configuration:")
print(
    "  PER_PROVIDER_CAP =",
    PER_PROVIDER_CAP
)
print(
    "  SKIP_GIANTS_OVER_MB =",
    SKIP_GIANTS_OVER_MB
)


# ============================================================
# INITIALISE THE PIPELINE
# ============================================================
# Confirm that the function definitions have loaded and display the corpus configuration before starting the expensive parts of the experiment.
print("\nLoading corpus...")
cards, report = load_corpus_cards()
n_prov = sum(1 for k, v in report.items() if (not k.startswith("_") and isinstance(v, int) and v > 0)) # Count providers that contributed searchable records.
print("\nCORPUS ACCOUNTING")
print(
    f"  Total corpus records:      "
    f"{report['_total_records']:,}"
)
print(
    f"  Searchable records:        "
    f"{report['_loaded_records']:,}"
)
print(
    f"  Excluded from retrieval:   "
    f"{report['_excluded_records']:,}"
)
print(
    f"  Accounting check:          "
    f"{report['_loaded_records'] + report['_excluded_records']:,}"
)
print(
    f"  Providers loaded:          "
    f"{n_prov}"
)
print(
    "  Skipped providers:",
    [
        k
        for k, v in report.items()
        if isinstance(v, str)
    ]
)


# ============================================================
# EXCLUSION REPORT
# ============================================================
# Show why records were excluded from retrieval so that the searchable corpus can be distinguished from the full corpus.
if report["_excluded_reasons"]:
    print("\nEXCLUSION REASONS:")
    for reason, count in sorted(report["_excluded_reasons"].items(), key=lambda x: x[1], reverse=True):
        print(f"  {reason}: {count:,}")


# ============================================================
# DUPLICATE RETRIEVAL-ID CHECK
# ============================================================
# Check whether multiple records share the same retrieval ID. Duplicate IDs could make it difficult to distinguish records correctly during retrieval and result fusion.
duplicates = report["_duplicate_record_ids"]
print("\nRETRIEVAL-ID COLLISION CHECK")

if duplicates:
    print(
        f"  ⚠️ Duplicate record IDs: "
        f"{len(duplicates):,}"
    )

    for rid, count in list(duplicates.items())[:10]:
        print(
            f"    {rid} "
            f"(additional occurrences: {count})"
        )
else:
    print("  ✅ No duplicate record IDs found.")


# ============================================================
# BUILD RETRIEVAL INDEXES
# ============================================================
# Build both retrieval systems: BM25 for keyword matching and the dense retriever for semantic similarity.
print("\nBuilding BM25...")
bm25 = BM25(cards)

print("Building dense (GPU)...")
dense = DenseRetriever(cards)


# ============================================================
# TEST QUERIES
# ============================================================
# Define several representative queries to exercise different combinations of activity, date, price, geographic, and evidence constraints.
queries = [
    (
        "swimming (Aug-Sep)",
        {
            "activity": "swimming",
            "date_from": "2026-08-01",
            "date_to": "2026-09-30"
        }
    ),
    (
        "FREE swimming",
        {
            "activity": "swimming",
            "free": True,
            "date_from": "2026-08-01",
            "date_to": "2026-09-30"
        }
    ),
    (
        "swimming in Camden",
        {
            "activity": "swimming",
            "region": "Camden",
            "date_from": "2026-08-01",
            "date_to": "2026-09-30"
        }
    ),
    (
        "beginner swimming (indeterminate test)",
        {
            "activity": "swimming",
            "level": "beginner",
            "date_from": "2026-08-01",
            "date_to": "2026-09-30"
        }
    )
]

# ============================================================
# PREFERENCE MODEL SETUP
# ============================================================
# Initialise the preference model and give it a simple starting preference for swimming-related activities for this experiment.
belief = PreferenceBelief()
belief.update("activity", "swim")

output_lines = []

for label, q in queries:
    print(f"\n{'=' * 70}")
    print(f"QUERY: {label}")
    print(f"{'=' * 70}")
    res = recommend(q, cards, bm25, dense, belief, 5)
    verdicts = Counter(r["gate"] for r in res)
    print("Verdicts:", dict(verdicts))

    for r in res:
        print(
            f"  {r['gate']:22s} "
            f"pref={r['preference']:.2f} "
            f"name={r['name']!r} "
            f"record_id={r['record_id']!r} "
            f"start={r['startDate']!r}"
        )

# ============================================================
# RUN THE TEST QUERIES
# ============================================================
# Run each test query through the complete retrieval, gating, and preference pipeline and record the resulting verdicts.
    output_lines.append(f"\n=== {label} ===")
    output_lines.append(f"verdicts: {dict(verdicts)}")

    for r in res:
        output_lines.append(
            f"{r['gate']:22s} "
            f"pref={r['preference']:.2f} "
            f"name={r['name']!r} "
            f"record_id={r['record_id']!r} "
            f"start={r['startDate']!r}"
        )


# ============================================================
# SAVE RESULTS
# ============================================================
# Save the corpus accounting and query results to a plain-text file so that the experiment can be reviewed later without rerunning the entire corpus pipeline.
full_output = (
    "D4 CORPUS RESULTS\n"
    "=================\n\n"
    f"Total corpus records: "
    f"{report['_total_records']:,}\n"
    f"Searchable records: "
    f"{report['_loaded_records']:,}\n"
    f"Excluded records: "
    f"{report['_excluded_records']:,}\n"
    f"Providers loaded: "
    f"{n_prov}\n\n"
    + "\n".join(output_lines)
)

output_path = (
    "/content/drive/MyDrive/"
    "d4_corpus_results.txt"
)

with open(output_path,"w",encoding="utf-8") as f:
    f.write(full_output)
    
print("\nResults saved to:",output_path)
print("\nDone.")

print(
    f"Corpus: {report['_total_records']:,} "
    f"total / "
    f"{report['_loaded_records']:,} searchable / "
    f"{report['_excluded_records']:,} excluded."
)

Cell 4 function definitions loaded.

Full corpus configuration:
  PER_PROVIDER_CAP = None
  SKIP_GIANTS_OVER_MB = None

Loading corpus...

CORPUS ACCOUNTING
  Total corpus records:      737,227
  Searchable records:        724,361
  Excluded from retrieval:   12,866
  Accounting check:          737,227
  Providers loaded:          33
  Skipped providers: []

EXCLUSION REASONS:
  no_searchable_text: 12,866

RETRIEVAL-ID COLLISION CHECK
  ✅ No duplicate record IDs found.

Building BM25...
Building dense (GPU)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  loaded cached embeddings

QUERY: swimming (Aug-Sep)
Verdicts: {'supported_match': 5}
  supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760741' start='2026-09-06T10:30:00+01:00'
  supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760738' start='2026-09-06T10:10:00+01:00'
  supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760737' start='2026-09-06T10:50:00+01:00'
  supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760740' start='2026-09-06T10:20:00+01:00'
  supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760739' start='

# Cell 5: D5 Facet Diagnostics
This cell checks the activity facets produced from the loaded corpus against the
activity vocabulary used by the D5 preference model.

It reports:
- how many records have each activity facet;
- which activity values the D5 model currently knows about; and
- any activity facets found in the corpus that are not represented in the D5 vocabulary.

In [ ]:
# Display a clear heading so the diagnostic output is easy to identify when reviewing the notebook results.
print("=" * 70)
print("D5 FACET DIAGNOSTIC")
print("=" * 70)

# Count how many times each facet value occurs across the loaded search cards.
# Each card can contain several facets, such as activity, day_type, and time. We count the key/value pairs separately.
facet_counts = Counter()

for c in cards:
    for key, value in c["facets"].items():
        facet_counts[(key, value)] += 1

# Report the activity facets found in the corpus. This shows the actual activity categories produced by the facet-building logic in Cell 4, together with their frequency.
print("\nACTIVITY FACETS:")
for (key, value), count in sorted(facet_counts.items()):
    if key == "activity":
        print(f"  {value!r}: {count:,}")

# Show the activity vocabulary currently configured in the D5 preference model.
# These are the activity values that the preference model can assign probabilities to.
print("\nD5 ACTIVITY VOCABULARY:")
print(list(belief.probs["activity"]))

# These are the activity values that the preference model can assign probabilities to.
# An unknown value does not necessarily mean the corpus is wrong. It indicates that the facet-building logic and the D5 preference vocabulary are not fully aligned for that value.
print("\nUNKNOWN ACTIVITY FACETS:")
for (key, value), count in sorted(facet_counts.items()):
    if key == "activity" and value not in belief.probs["activity"]:
        print(f"  {value!r}: {count:,}")

D5 FACET DIAGNOSTIC

ACTIVITY FACETS:
  'fitness suite': 15,003
  'group exercise': 9,892
  'leisure pool': 9
  'swim': 154,054

D5 ACTIVITY VOCABULARY:
['swim', 'fitness suite', 'group exercise', 'junior', 'leisure pool']

UNKNOWN ACTIVITY FACETS:


# Cell 6: Inspect Price Evidence for Free Swimming
This cell performs a final manual check of the evidence used for the "free swimming" query. It reruns the recommendation using the same constraints as the earlier test, then displays the underlying `offers` field for each returned record. This makes it possible to compare the gate verdict with the actual price information stored in the corpus.

In [ ]:
# Display a clear heading so this QA output is easy to identify when reviewing the notebook results.
print("\n" + "=" * 70)
print("FREE SWIMMING PRICE EVIDENCE")
print("=" * 70)

# Define the test query: We are looking specifically for swimming sessions that are free and take place between 1 August and 30 September 2026.
q = {
    "activity": "swimming",
    "free": True,
    "date_from": "2026-08-01",
    "date_to": "2026-09-30"
}

# Run the query through the same recommendation pipeline used earlier in the notebook.
# A fresh PreferenceBelief is used here because this QA check is focused on the evidence and gating result rather than personalised preference ranking.
res = recommend(q, cards, bm25, dense, PreferenceBelief(), top_k=5)

# Create a lookup from record ID to the original search card. This lets us move from the recommendation result back to the
# complete underlying corpus record so that the original "offers" field can be inspected.
by_id = {c["record_id"]: c for c in cards}

# Inspect each returned result. For every candidate we display:
#   - the session name;
#   - its corpus record ID;
#   - the evidence-gating verdict; and
#   - the original offers/price data used for inspection.
for r in res:
    card = by_id[r["record_id"]]
    fields = card["_record"].get("fields", {}) or {}
    offers = fields.get("offers")

# Showing the raw offers field makes the QA result auditable against the underlying corpus data rather than relying only on the final verdict.
    print("\nNAME:", r["name"])
    print("RECORD:", r["record_id"])
    print("VERDICT:", r["gate"])
    print("OFFERS:", offers)


FINAL QA — FREE SWIMMING PRICE EVIDENCE

NAME: Lane Swimming
RECORD: https://ledleisure.gs-signature.cloud/OpenActive/api/scheduled-sessions/100156164
VERDICT: bounded_non_match
OFFERS: [{'@type': 'Offer', 'identifier': 'JRNM', 'name': 'Junior U16', 'description': 'Swim', 'acceptedPaymentMethod': ['http://purl.org/goodrelations/v1#Cash', 'http://purl.org/goodrelations/v1#PaymentMethodCreditCard'], 'ageRestriction': {'@type': 'QuantitativeValue', 'maxValue': 15}, 'price': 4.1, 'priceCurrency': 'GBP'}, {'@type': 'Offer', 'identifier': 'STAND', 'name': 'Adult 16+', 'description': 'Swim', 'acceptedPaymentMethod': ['http://purl.org/goodrelations/v1#Cash', 'http://purl.org/goodrelations/v1#PaymentMethodCreditCard'], 'price': 6.85, 'priceCurrency': 'GBP'}]

NAME: Swim For All
RECORD: https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760741
VERDICT: bounded_non_match
OFFERS: [{'@type': 'Offer', 'name': 'Adult', 'acceptedPaymentMethod': ['http://purl.org/goodrelat

# Cell 7: Inspect Location Evidence for Camden
This cell performs a final manual check of the evidence used for the "swimming in Camden" query. It reruns the recommendation with a Camden location constraint, then displays the underlying `location` field for each returned record. This allows the final gating verdict to be compared with the location information stored in the original corpus record.

In [ ]:
# Display a clear heading so this QA output is easy to identify when reviewing the notebook results.
print("\n" + "=" * 70)
print("CAMDEN LOCATION EVIDENCE")
print("=" * 70)

# Define the test query. We are looking for swimming sessions in Camden taking place between 1 August and 30 September 2026.
q = {
    "activity": "swimming",
    "region": "Camden",
    "date_from": "2026-08-01",
    "date_to": "2026-09-30"
}

# Same as the above cell
res = recommend(q, cards, bm25, dense, PreferenceBelief(), top_k=5)

# Same as the above cell
by_id = {c["record_id"]: c for c in cards}

# Similar to the above cell
for r in res:
    card = by_id[r["record_id"]]
    fields = card["_record"].get("fields", {}) or {}
    print("\nNAME:", r["name"])
    print("RECORD:", r["record_id"])
    print("VERDICT:", r["gate"])
    print("LOCATION:", fields.get("location"))


FINAL QA — CAMDEN LOCATION EVIDENCE

NAME: Lane Swimming
RECORD: https://ledleisure.gs-signature.cloud/OpenActive/api/scheduled-sessions/100156164
VERDICT: bounded_non_match
LOCATION: {'@type': 'Place', 'identifier': 'EDSP', 'name': 'Sidmouth Swimming Pool', 'description': "LED's Sidmouth\xa0Swimming Pool\xa0is light, bright, airy and based in the centre of this iconic Devon seaside town\nWe run a thorough and varied programme throughout the year to ensure that there is always something to keep you wanting to get in the water!\nFirst time splashes for beginners and improvement sessions for the more seasoned swimmers, whatever your level why not put on your goggles and come on in!\n", 'address': {'@type': 'PostalAddress', 'addressCountry': 'GB', 'addressLocality': 'Sidmouth', 'addressRegion': 'Devon', 'postalCode': 'EX10 8XR', 'streetAddress': 'Ham Lane'}, 'amenityFeature': [{'@type': 'ChangingFacilities', 'name': 'Changing Facilities', 'value': True}, {'@type': 'Showers', 'name': 'Sho

# Cell 8: Activity `prefLabel` Gating
This cell checks the activity matching rule against real corpus records. It performs two checks:
1. finds real records whose activity `prefLabel` contains "swim" and confirms
   that the activity matcher accepts them;
2. finds a clearly non-swimming activity and confirms that it is rejected.

The check uses the same `prefLabel`-only activity matching logic used by the
retrieval and gating pipeline.

In [ ]:
# Display a clear heading so this QA output is easy to identify when reviewing the notebook results.
print("=" * 70)
print("ACTIVITY PREFLABEL GATING")
print("=" * 70)

# Find a real swimming record from the existing corpus
swim_cards = []

# We inspect the activity prefLabels rather than relying on the record's category or another derived field. This checks the
for c in cards:
    activity_labels = _pref_labels(c["_record"].get("fields", {}).get("activity"))
    labels = [str(x)for x in activity_labels]

    if any("swim" in _norm_text(label) for label in labels):
        swim_cards.append(c)
        # We only need a few examples for this manual QA check, so the search stops after finding three matching records.
        if len(swim_cards) >= 3:
            break

print(f"\nFound real swimming records: {len(swim_cards)}")

# Test the activity matcher against each real swimming record. The raw prefLabels are printed alongside the matcher result
# so that the QA output shows both the underlying evidence and the resulting boolean decision.
for c in swim_cards:
    activity_labels = _pref_labels(
        c["_record"].get("fields", {}).get("activity")
    )
    print("\nNAME:", c["name"])
    print("RECORD:", c["record_id"])
    print("ACTIVITY prefLabels:", activity_labels)
    pred = activity_matches("swimming")
    actual = pred(c["_record"].get("fields", {}).get("activity"))
    print("activity_matches('swimming'):", actual)


# Check that a clearly non-swimming activity does NOT match.
print("\n" + "-" * 70)
print("NON-SWIMMING NEGATIVE CHECK")
print("-" * 70)

non_swim = None

# Find a real record whose activity prefLabels do not contain "swim", then confirm that the swimming matcher rejects it.
for c in cards:
    activity_labels = _pref_labels(c["_record"].get("fields", {}).get("activity"))
    labels = [str(x) for x in activity_labels]
    if labels and not any("swim" in _norm_text(label) for label in labels):
        non_swim = c
        break
# This provides a simple negative case to check that the matcher is not accepting unrelated activity values.

# Report the negative-check result. If no suitable non-swimming record exists, the test is reported as inconclusive rather than being treated as a pass.
if non_swim is None:
    print("Could not find a non-swimming activity record.")
else:
    activity_labels = _pref_labels(non_swim["_record"].get("fields", {}).get("activity"))
    actual = activity_matches("swimming")(non_swim["_record"].get("fields", {}).get("activity"))

    print("NAME:", non_swim["name"])
    print("RECORD:", non_swim["record_id"])
    print("ACTIVITY prefLabels:", activity_labels)
    print("activity_matches('swimming'):", actual)

# Give a simple final status for the negative check. A false result confirms that the selected non-swimming record is rejected by the swimming activity matcher.
    if actual is False:
        print("\n✅ ACTIVITY GATE PASSED")
        print("   Swimming matches swimming prefLabels.")
        print("   Non-swimming prefLabels do not match.")
    else:
        print("\n❌ ACTIVITY GATE FAILED")

FINAL QA — ACTIVITY PREFLABEL GATING

Found real swimming records: 3

NAME: Swim For All
RECORD: https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:24320070
ACTIVITY prefLabels: ['Swimming']
activity_matches('swimming'): True

NAME: Swim For All
RECORD: https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:24320080
ACTIVITY prefLabels: ['Swimming']
activity_matches('swimming'): True

NAME: Swim For All
RECORD: https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:24320180
ACTIVITY prefLabels: ['Swimming']
activity_matches('swimming'): True

----------------------------------------------------------------------
NON-SWIMMING NEGATIVE CHECK
----------------------------------------------------------------------
NAME: Soft Play
RECORD: https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:28692768
ACTIVITY prefLabels: ['Soft Play']
activity_matches('swimming'): False

✅ ACTIVITY GATE PASSED
  

# Cell 9: Hybrid Retrieval Sanity Check
This cell checks the behaviour of the hybrid retrieval pipeline using a
small set of representative queries. For each query, it compares:
- BM25 lexical retrieval;
- dense semantic retrieval; and
- the hybrid ranking produced by Reciprocal Rank Fusion (RRF).

The diagnostic reports how much the two retrieval methods overlap and
whether the hybrid top-10 contains any records outside their individual
top-10 results. 

This is a retrieval sanity check rather than a judgement of whether
each returned result is semantically correct.

In [ ]:
print("=" * 70)
print("FINAL QA — HYBRID RETRIEVAL SANITY CHECK")
print("=" * 70)

# Define a small set of representative queries.
# The aim is to exercise both lexical and semantic retrieval rather than testing only the exact word "swimming".
qa_queries = ["swimming", "lane swim", "beginner fitness class", "water aerobics for seniors", "kids activities", "gentle exercise",]

# Run each query through both retrieval methods and then combine their rankings using Reciprocal Rank Fusion.
# Each method returns its top 10 results so that the outputs can be compared consistently.
for q in qa_queries:
    bm = bm25.search(q, 10)
    dn = dense.search(q, 10)
    hy = rrf([bm, dn], top_k=10)
    
    # Convert each result list to a set of record IDs. The IDs allow us to compare which records were returned by each retrieval method without considering their rank.
    bm_ids = {x[0] for x in bm}
    dn_ids = {x[0] for x in dn}
    hy_ids = {x for x, _ in hy}
    
    # Count the overlap between BM25 and dense retrieval. A higher overlap means the two retrieval approaches are returning more of the same records for this query.
    overlap = len(bm_ids & dn_ids)
    
    # Check whether the hybrid top-10 contains records that were not present in either individual top-10. With the current RRF implementation, this should normally
    # be zero when RRF is combining only the supplied top-10 rankings.
    hybrid_from_deeper = len(hy_ids - (bm_ids | dn_ids))

    # Print the diagnostic information for this query.
    print("\n" + "=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)

    print(f"BM25 top10:   {len(bm)} results")
    print(f"Dense top10:  {len(dn)} results")
    print(f"Hybrid top10: {len(hy)} results")

    print(
        f"BM25 & Dense overlap: "
        f"{overlap}/10"
    )

    print(
        f"Hybrid results not in either top10: "
        f"{hybrid_from_deeper}"
    )

# Basic pipeline check. A successful 10/10/10 result means each retrieval stage returned the requested number of candidates. It does not mean that the candidates are necessarily relevant.
    if (len(bm) == 10 and len(dn) == 10 and len(hy) == 10):
        print("✅ Retrieval pipeline returned 10/10/10")
    else:
        print("❌ Retrieval pipeline did not return 10/10/10")

FINAL QA — HYBRID RETRIEVAL SANITY CHECK

QUERY: swimming
BM25 top10:   10 results
Dense top10:  10 results
Hybrid top10: 10 results
BM25 & Dense overlap: 0/10
Hybrid results not in either top10: 0
✅ Retrieval pipeline returned 10/10/10

QUERY: lane swim
BM25 top10:   10 results
Dense top10:  10 results
Hybrid top10: 10 results
BM25 & Dense overlap: 0/10
Hybrid results not in either top10: 0
✅ Retrieval pipeline returned 10/10/10

QUERY: beginner fitness class
BM25 top10:   10 results
Dense top10:  10 results
Hybrid top10: 10 results
BM25 & Dense overlap: 0/10
Hybrid results not in either top10: 0
✅ Retrieval pipeline returned 10/10/10

QUERY: water aerobics for seniors
BM25 top10:   10 results
Dense top10:  10 results
Hybrid top10: 10 results
BM25 & Dense overlap: 3/10
Hybrid results not in either top10: 0
✅ Retrieval pipeline returned 10/10/10

QUERY: kids activities
BM25 top10:   10 results
Dense top10:  10 results
Hybrid top10: 10 results
BM25 & Dense overlap: 0/10
Hybrid results n

# Cell 10: End-to-End Gating Summary

This cell provides a final sanity check of the retrieval and evidence-gating
pipeline using four representative queries.The tests cover:
- a positive swimming match;
- a free-swimming query expected to produce a bounded non-match;
- a Camden-constrained query expected to produce a bounded non-match; and
- a beginner-swimming query expected to exercise the indeterminate state.

For each test, the expected verdict is compared with the verdicts actually
returned by the recommendation pipeline.

A test passes when the expected verdict is observed in the returned results.
This confirms that the relevant terminal state is being produced, but does
not by itself prove that every returned candidate has that verdict.

In [ ]:
# Formatting like above
print("=" * 70)
print("END-TO-END GATING SUMMARY")
print("=" * 70)

# Define test cases like the cells above
final_tests = [
    (
        "POSITIVE — swimming",
        {
            "activity": "swimming",
            "date_from": "2026-08-01",
            "date_to": "2026-09-30",
        },
        "supported_match",
    ),
    (
        "NEGATIVE — free swimming",
        {
            "activity": "swimming",
            "free": True,
            "date_from": "2026-08-01",
            "date_to": "2026-09-30",
        },
        "bounded_non_match",
    ),
    (
        "GATE NEGATIVE — Camden candidate set",
        {
            "activity": "swimming",
            "region": "Camden",
            "date_from": "2026-08-01",
            "date_to": "2026-09-30",
        },
        "bounded_non_match",
    ),
    (
        "INDETERMINATE — beginner swimming",
        {
            "activity": "swimming",
            "level": "beginner",
            "date_from": "2026-08-01",
            "date_to": "2026-09-30",
        },
        "evidence_indeterminate",
    ),
]

# Track whether every test observed its expected verdict. This starts as True and is changed to False if any expected verdict is missing from its returned results.
all_passed = True

# Run each final QA query through the full recommendation pipeline.
# This means the test exercises retrieval, candidate lookup, constraint construction, evidence reduction and final ordering.
for label, query, expected in final_tests:
    res = recommend(query, cards, bm25, dense, belief, 5,)
    # Count the terminal verdicts returned by the gate. This makes the output easier to inspect than printing every individual recommendation.
    verdicts = Counter(r["gate"] for r in res)

    # Display the expected and observed results for this test.
    print("\n" + "-" * 70)
    print(label)
    print("-" * 70)

    print("Expected:", expected)
    print("Actual:", dict(verdicts))

    # A test passes when the expected terminal verdict appears in the returned results. If it is absent, mark the overall QA run as failed.
    if expected in verdicts:
        print("✅ PASS")
    else:
        print("❌ FAIL")
        all_passed = False


# Print the overall QA result. This summarises whether all four expected terminal states were observed during the final test run.
print("\n" + "=" * 70)

if all_passed:
    print("✅ FINAL END-TO-END QA PASSED")
    print("The retrieval + gating pipeline behaves as expected.")
else:
    print("❌ FINAL END-TO-END QA FAILED")
    print("At least one expected verdict was not observed.")

print("=" * 70)

FINAL QA — END-TO-END GATING SUMMARY

----------------------------------------------------------------------
POSITIVE — swimming
----------------------------------------------------------------------
Expected: supported_match
Actual: {'supported_match': 5}
✅ PASS

----------------------------------------------------------------------
NEGATIVE — free swimming
----------------------------------------------------------------------
Expected: bounded_non_match
Actual: {'bounded_non_match': 5}
✅ PASS

----------------------------------------------------------------------
GATE NEGATIVE — Camden candidate set
----------------------------------------------------------------------
Expected: bounded_non_match
Actual: {'bounded_non_match': 5}
✅ PASS

----------------------------------------------------------------------
INDETERMINATE — beginner swimming
----------------------------------------------------------------------
Expected: evidence_indeterminate
Actual: {'evidence_indeterminate': 5}
✅ PA

# Cell 11: D5 Integration Diagnostic

This cell checks the handoff between the D4 retrieval/gating pipeline and the D5
preference model.

It runs a representative swimming query and prints the resulting gate,
preference score, and activity name for each returned record.

The purpose is to confirm that D5 receives the results produced by the
retrieval and evidence-gating stages and attaches preference scores correctly.
Code cell

In [ ]:
# Define the test queries
test_query = {
    "activity": "swimming",
    "date_from": "2026-08-01",
    "date_to": "2026-09-30",
}

# This combines retrieval, evidence gating, and preference scoring to produce the final ranked results.
results = recommend(test_query, cards, bm25,dense,belief,5,)

# Display the D5 Integration Results. For each returned record, show the evidence-gating verdict, D5 preference score, and human-readable activity name.
print("=" * 70)
print("D5 INTEGRATION DIAGNOSTIC")
print("=" * 70)
for r in results:
    print("gate=", r.get("gate"), "preference=", r.get("preference"), "name=", r.get("name"))

D5 INTEGRATION DIAGNOSTIC
gate= supported_match preference= 0.561 name= Swim For All
gate= supported_match preference= 0.561 name= Swim For All
gate= supported_match preference= 0.561 name= Swim For All
gate= supported_match preference= 0.561 name= Swim For All
gate= supported_match preference= 0.561 name= Swim For All


# Cell 12: D4 Ablation Test (BM25 vs Dense vs RRF)

Compare the results produced by three retrieval approaches:
1. BM25 lexical retrieval
2. Dense semantic retrieval
3. Hybrid retrieval using Reciprocal Rank Fusion (RRF)

The comparison is made at two levels:
- Record level: whether the same record IDs are retrieved
- Name level: whether the same activity/session names are surfaced

The name-level comparison is useful because the corpus may contain multiple records for the same activity. Two retrieval methods can therefore return different records while still surfacing the same underlying activity.

The hybrid method uses deeper BM25 and dense rankings (top 100) before producing the final top-k results. This allows RRF to promote records that perform reasonably well across both retrieval methods.

In [ ]:
# ============================================================
# D4 ABLATION — BM25 vs Dense vs Hybrid (RRF)
#
# Compares retrieval at two levels:
#   1. record-level: exact record_id overlap
#   2. name-level: unique activity/session names surfaced
#
# This avoids interpreting duplicate instances of the same
# activity as completely different retrieval classes.
# ============================================================
from collections import Counter

# Compare BM25, dense retrieval and their RRF hybrid for a set of test queries.
def ablation(queries, cards, bm25, dense, top_k=10):
    # Create a lookup from record ID to the full search card. This lets us recover activity/session names after retrieval.
    by_id = {c["record_id"]: c for c in cards}

    # Run the comparison separately for each test query.
    for qt in queries:
        # INDIVIDUAL RETRIEVAL METHODS
        # First collect the top-k results from BM25 and dense retrieval independently.
        r2 = [iid for iid, _ in bm25.search(qt, top_k)]
        rd = [iid for iid, _ in dense.search(qt, top_k)]
        
        # HYBRID RETRIEVAL
        # Use deeper top-100 rankings from both methods before applying Reciprocal Rank Fusion. This gives RRF more candidates to combine than the final top-k alone.
        r2_full = bm25.search(qt, 100)
        rd_full = dense.search(qt, 100)
        fused = rrf([r2_full, rd_full], top_k=top_k)
        rh = [iid for iid, _ in fused]

        # Convert the three result lists to sets so that overlap and method-specific results can be compared directly.
        set2 = set(r2)
        setd = set(rd)
        seth = set(rh)

        # RECORD-LEVEL COMPARISON
        # Compare the exact record IDs returned by BM25 and dense retrieval. This shows how much the two methods agree about individual corpus records.
        both = set2 & setd
        only_bm25 = set2 - setd
        only_dense = setd - set2
        print(f"\n{'=' * 70}")
        print(f"QUERY: '{qt}'")
        print(f"{'=' * 70}")
        print(f"  BM25 top{top_k}:   {len(set2)} results")
        print(f"  Dense top{top_k}:  {len(setd)} results")
        print(f"  Hybrid top{top_k}: {len(seth)} results")
        print(f"\n  RECORD-LEVEL OVERLAP")
        print(
            f"  BM25 & Dense: "
            f"{len(both)}/{top_k} "
            f"({len(both)/top_k:.0%} agreement)"
        )
        print(f"  BM25-only:   {len(only_bm25)}")
        print(f"  Dense-only:  {len(only_dense)}")

        # UNIQUE-NAME COMPARISON
        # Compare the names represented by each result set. This avoids treating different instances of the same activity as completely different retrieval outcomes.
        names_bm25 = {by_id[i]["name"] for i in set2 if i in by_id}
        names_dense = {by_id[i]["name"] for i in setd if i in by_id}
        names_hybrid = {by_id[i]["name"] for i in seth if i in by_id}
        # Identify names returned by both methods and names unique to either BM25 or dense retrieval.
        names_both = names_bm25 & names_dense
        names_only_bm25 = names_bm25 - names_dense
        names_only_dense = names_dense - names_bm25

        print(f"\n  NAME-LEVEL COMPARISON")
        print(
            f"  Unique names — BM25:  "
            f"{len(names_bm25)}"
        )
        print(
            f"  Unique names — Dense: "
            f"{len(names_dense)}"
        )
        print(
            f"  Unique names — Hybrid: "
            f"{len(names_hybrid)}"
        )
        print(
            f"  Name overlap: "
            f"{len(names_both)} "
            f"(of {max(len(names_bm25), len(names_dense))} "
            f"unique names)"
        )

        # Show examples of names that dense retrieval found but BM25 did not, which can indicate semantic retrieval behaviour beyond exact lexical matching.
        if names_only_dense:
            print(
                f"  --> Dense unique names "
                f"BM25 missed: "
                f"{sorted(names_only_dense)[:10]}"
            )
        # Show the equivalent examples for BM25.
        if names_only_bm25:
            print(
                f"  --> BM25 unique names "
                f"Dense missed: "
                f"{sorted(names_only_bm25)[:10]}"
            )

        # HYBRID NOVELTY
        # Check whether RRF promoted any records into its final top-k that were not present in either method's own top-k list.
        hybrid_new = (seth - (set2 | setd))
        print(
            f"\n  Hybrid promoted into top{top_k} "
            f"from deeper ranks: "
            f"{len(hybrid_new)}"
        )
        
        # If RRF promoted new records, show their corresponding activity/session names for easier interpretation.
        if hybrid_new:
            hybrid_new_names = {by_id[i]["name"] for i in hybrid_new if i in by_id}
            print(
                f"  Hybrid-new names: "
                f"{sorted(hybrid_new_names)[:10]}"
            )
        else:
            print("  Hybrid-new names: []")


if __name__ == "__main__":
# Run the ablation study when this cell is executed directly. These queries were chosen to expose differences between lexical matching and semantic matching:
#   - literal terms
#   - phrase variations
#   - semantic/paraphrase queries
#   - queries using different terminology
    ablation_queries = [
        "swimming",
        "lane swim",
        "beginner fitness class",
        "water aerobics for seniors",
        "kids activities",
        "gentle exercise",
    ]

    ablation(
        ablation_queries,
        cards,
        bm25,
        dense,
        top_k=10
    )


QUERY: 'swimming'
  BM25 top10:   10 results
  Dense top10:  10 results
  Hybrid top10: 10 results

  RECORD-LEVEL OVERLAP
  BM25 & Dense: 0/10 (0% agreement)
  BM25-only:   10
  Dense-only:  10

  NAME-LEVEL COMPARISON
  Unique names — BM25:  2
  Unique names — Dense: 1
  Unique names — Hybrid: 3
  Name overlap: 0 (of 2 unique names)
  --> Dense unique names BM25 missed: ['Swim For All']
  --> BM25 unique names Dense missed: ['Lane Swimming', 'Public Swimming']

  Hybrid promoted into top10 from deeper ranks: 0
  Hybrid-new names: []

QUERY: 'lane swim'
  BM25 top10:   10 results
  Dense top10:  10 results
  Hybrid top10: 10 results

  RECORD-LEVEL OVERLAP
  BM25 & Dense: 0/10 (0% agreement)
  BM25-only:   10
  Dense-only:  10

  NAME-LEVEL COMPARISON
  Unique names — BM25:  2
  Unique names — Dense: 2
  Unique names — Hybrid: 4
  Name overlap: 0 (of 2 unique names)
  --> Dense unique names BM25 missed: ['Lane Swim', 'Lane Swim - Medium']
  --> BM25 unique names Dense missed: ['Lane 

# Cell 13: Final Output Check
This final check confirms that the corpus results file was successfully written to Google Drive.

If the file exists, the cell reports its size and displays the first 4,000 characters as a quick preview. This provides a simple sanity check that the previous corpus run completed and produced readable output.

In [ ]:
from pathlib import Path

# Locate the results file written by the corpus analysis.
p = Path("/content/drive/MyDrive/d4_corpus_results.txt")

print("Exists:", p.exists())    # Check whether the expected output file exists.
# Report the file size when the file is available. This gives a quick indication that some output was written.
print("Size:", p.stat().st_size if p.exists() else None)

# Preview the beginning of the saved results. Limiting the preview to 4,000 characters keeps the notebook
# output manageable while still allowing the saved results to be inspected for obvious problems.
if p.exists():
    print("\n--- FILE PREVIEW ---")
    print(p.read_text(encoding="utf-8")[:4000])

Exists: True
Size: 4131

--- FILE PREVIEW ---
D4 CORPUS RESULTS

Total corpus records: 737,227
Searchable records: 724,361
Excluded records: 12,866
Providers loaded: 33


=== swimming (Aug-Sep) ===
verdicts: {'supported_match': 5}
supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760741' start='2026-09-06T10:30:00+01:00'
supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760738' start='2026-09-06T10:10:00+01:00'
supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760737' start='2026-09-06T10:50:00+01:00'
supported_match        pref=0.56 name='Swim For All' record_id='https://better-admin.org.uk/api/openactive/better/scheduled-sessions/activity:89760740' start='2026-09-06T10:20:00+01:00'
supported_match        pr